In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [9]:
from dotenv import load_dotenv

load_dotenv() 

llm= HuggingFaceEndpoint(
    repo_id='openai/gpt-oss-120b',
    task='text-generation'
)
llm = ChatHuggingFace(llm=llm)

In [3]:
documents = [

    Document(
        page_content="""
        Reducing API response latency often requires identifying operations
        that take the most time. Developers should measure request duration,
        database execution time, network latency, and application processing
        time before deciding where optimization is needed.
        """,
        metadata={
            "topic": "performance",
            "concept": "latency",
            "id": "perf_01"
        }
    ),

    Document(
        page_content="""
        Caching is an effective technique for speeding up applications.
        Frequently requested information can be stored in memory so that
        subsequent requests avoid expensive database operations. Redis is
        commonly used as a distributed cache.
        """,
        metadata={
            "topic": "performance",
            "concept": "caching",
            "id": "perf_02"
        }
    ),

    Document(
        page_content="""
        Database performance can be improved by creating appropriate indexes,
        optimizing SQL queries, avoiding unnecessary joins, and using database
        connection pools. Query execution plans can help identify inefficient
        operations.
        """,
        metadata={
            "topic": "performance",
            "concept": "database",
            "id": "perf_03"
        }
    ),

    Document(
        page_content="""
        Scalability can be improved by distributing incoming traffic across
        multiple application servers. Load balancers such as Nginx can route
        requests between backend instances and prevent a single server from
        becoming overloaded.
        """,
        metadata={
            "topic": "performance",
            "concept": "load_balancing",
            "id": "perf_04"
        }
    ),

    Document(
        page_content="""
        Long-running operations should not always be executed synchronously
        during an HTTP request. Background workers and message queues allow
        expensive tasks to execute asynchronously while the API responds
        quickly to the client.
        """,
        metadata={
            "topic": "performance",
            "concept": "async_processing",
            "id": "perf_05"
        }
    ),

    Document(
        page_content="""
        Horizontal scaling involves running multiple instances of an
        application instead of relying on a single powerful machine. Traffic
        can be distributed among these instances, allowing the system to
        support more concurrent requests.
        """,
        metadata={
            "topic": "performance",
            "concept": "horizontal_scaling",
            "id": "perf_06"
        }
    ),

    Document(
        page_content="""
        API throughput can be improved by reducing unnecessary data transfer.
        Pagination limits the number of records returned in a response, while
        compression reduces the size of HTTP responses sent over the network.
        """,
        metadata={
            "topic": "performance",
            "concept": "api_optimization",
            "id": "perf_07"
        }
    )
]

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS Vector store from documents
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7892.10it/s]


In [6]:
# Create retrievers

similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={'k':5})

In [10]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k':5}),
    llm = llm
)

In [11]:
query = "How can I make my application faster?"

In [12]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [13]:
for i, doc in enumerate(similarity_results):
    print(f"\n ------Results {i+1} ------")
    print(f"Content:\n {doc.page_content}")


 ------Results 1 ------
Content:
 
        Reducing API response latency often requires identifying operations
        that take the most time. Developers should measure request duration,
        database execution time, network latency, and application processing
        time before deciding where optimization is needed.
        

 ------Results 2 ------
Content:
 
        Database performance can be improved by creating appropriate indexes,
        optimizing SQL queries, avoiding unnecessary joins, and using database
        connection pools. Query execution plans can help identify inefficient
        operations.
        

 ------Results 3 ------
Content:
 
        Caching is an effective technique for speeding up applications.
        Frequently requested information can be stored in memory so that
        subsequent requests avoid expensive database operations. Redis is
        commonly used as a distributed cache.
        

 ------Results 4 ------
Content:
 
        API throughp

In [14]:
for i, doc in enumerate(multiquery_results):
    print(f"\n ------Results {i+1} ------")
    print(f"Content:\n {doc.page_content}")


 ------Results 1 ------
Content:
 
        Database performance can be improved by creating appropriate indexes,
        optimizing SQL queries, avoiding unnecessary joins, and using database
        connection pools. Query execution plans can help identify inefficient
        operations.
        

 ------Results 2 ------
Content:
 
        Reducing API response latency often requires identifying operations
        that take the most time. Developers should measure request duration,
        database execution time, network latency, and application processing
        time before deciding where optimization is needed.
        

 ------Results 3 ------
Content:
 
        Caching is an effective technique for speeding up applications.
        Frequently requested information can be stored in memory so that
        subsequent requests avoid expensive database operations. Redis is
        commonly used as a distributed cache.
        

 ------Results 4 ------
Content:
 
        Scalability 